Ініціалізуємо необхідні бібліотеки.

In [ ]:
import sys
import numpy as np
from PyQt6.QtCore import Qt
from PyQt6.QtWidgets import QApplication, QCheckBox, QGridLayout, QGroupBox, QHBoxLayout, QLabel, QMainWindow, QPushButton, QSlider, QVBoxLayout, QWidget
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
from scipy.signal import iirfilter, sosfiltfilt

Реалізуйте функцію harmonic_with_noise.

In [ ]:
def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise, duration=5.0, discretization_rate=1000, base_noise=None):
    t = np.linspace(0, duration, int(duration * discretization_rate), endpoint=False)

    omega = 2 * np.pi * frequency
    y_clean = amplitude * np.sin(omega * t + phase)

    if base_noise is None:
        base_noise = np.random.normal(0.0, 1.0, size=t.shape)

    noise_std = np.sqrt(max(noise_covariance, 0.0))
    noise = noise_mean + noise_std * base_noise

    y_noisy = y_clean + noise

    y_display = y_noisy if show_noise else y_clean

    return t, y_clean, y_noisy, y_display

Отриману гармоніку з накладеним на неї шумом відфільтруйте за допомогою фільтру.

In [ ]:
def apply_iir_lowpass(signal, cutoff_frequency, sample_rate, filter_order):
    nyquist = sample_rate / 2.0

    safe_cutoff = float(np.clip(cutoff_frequency, 0.1, nyquist - 0.1))

    sos = iirfilter(N=int(filter_order), Wn=safe_cutoff, btype="lowpass", ftype="butter", output="sos", fs=sample_rate)

    filtered = sosfiltfilt(sos, signal)
    return filtered

У програмі має бути створено головне вікно з такими елементами інтерфейсу:

Створено клас для слайдерів та їх лейблів.

In [ ]:
class LabeledSlider(QWidget):
    def __init__(self, title, min_value, max_value, default_value, scale=100, decimals=2):
        super().__init__()

        self.scale = scale
        self.decimals = decimals

        self.title_label = QLabel(title)
        self.value_label = QLabel()
        self.value_label.setMinimumWidth(70)

        self.slider = QSlider(Qt.Orientation.Horizontal)
        self.slider.setRange(min_value, max_value)
        self.slider.setValue(default_value)

        top_layout = QHBoxLayout()
        top_layout.addWidget(self.title_label)
        top_layout.addStretch()
        top_layout.addWidget(self.value_label)

        main_layout = QVBoxLayout(self)
        main_layout.addLayout(top_layout)
        main_layout.addWidget(self.slider)

        self.slider.valueChanged.connect(self.update_label)
        self.update_label(self.slider.value())

    def update_label(self, raw_value):
        value = raw_value / self.scale
        self.value_label.setText(f"{value:.{self.decimals}f}")

    def value(self):
        return self.slider.value() / self.scale

    def set_value(self, real_value):
        self.slider.setValue(int(round(real_value * self.scale)))

Створено клас для полотна.

In [ ]:
class PlotCanvas(FigureCanvas):
    def __init__(self):
        self.figure = Figure(figsize=(11, 4.8), tight_layout=True)
        self.ax_original = self.figure.add_subplot(121)
        self.ax_filtered = self.figure.add_subplot(122)
        super().__init__(self.figure)

    def draw_plots(self, t, y_clean, y_noisy, y_display, y_filtered, show_noise, filter_enabled):
        self.ax_original.clear()
        self.ax_filtered.clear()

        #Через чекбокс користувач може вмикати або вимикати відображення шуму на графіку. Якщо прапорець прибрано – відображати «чисту гармоніку», якщо ні – зашумлену.
        if show_noise:
            self.ax_original.plot(t, y_display, label="Зашумлена гармоніка", color="orange")
            self.ax_original.plot(t, y_clean, label="Чиста гармоніка", color="blue", linewidth=2)
        else:
            self.ax_original.plot(t, y_clean, label="Чиста гармоніка", color="blue", linewidth=2)

        #Поле для графіка функції (plot) (або декілька полів)
        self.ax_original.set_title("Початковий сигнал")
        self.ax_original.set_xlabel("Час, с")
        self.ax_original.set_ylabel("Амплітуда")
        self.ax_original.grid(True)
        self.ax_original.legend()

        if filter_enabled:
            self.ax_filtered.plot(t, y_filtered, label="Після IIR-фільтра", color="purple")
            self.ax_filtered.set_title("Відфільтрований сигнал")
        else:
            self.ax_filtered.plot(t, y_noisy, label="Фільтр вимкнено", color="gray")
            self.ax_filtered.set_title("Фільтр вимкнено")

        self.ax_filtered.plot(t, y_clean, label="Чиста гармоніка", color="blue", linewidth=2)
        self.ax_filtered.set_xlabel("Час, с")
        self.ax_filtered.set_ylabel("Амплітуда")
        self.ax_filtered.grid(True)
        self.ax_filtered.legend()

        self.draw()

Клас вікна, де ми виводимо весь інтерфейс.

In [ ]:
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Гармоніка з шумом та IIR-фільтрацією")

        #Програма повинна мати початкові значення кожного параметру, а також передавати  параметри для відображення оновленого графіка.
        self.default_amplitude = 1.0
        self.default_frequency = 2.0
        self.default_phase = 0.0
        self.default_noise_mean = 0.0
        self.default_noise_covariance = 0.05
        self.default_show_noise = True
        self.default_filter_enabled = True
        self.default_cutoff_frequency = 5.0
        self.default_filter_order = 4

        self.duration = 5.0
        self.discretization_rate = 1000
        self.signal_length = int(self.duration * self.discretization_rate)

        self.rng = np.random.default_rng(42)
        #Зауваження: якщо ви змінили параметри гармоніки, але не змінювали параметри шуму, то шум має залишитись таким як і був, а не генеруватись наново. Якщо ви змінили параметри шуму, змінюватись має лише шум – параметри гармоніки мають залишатись незмінними.

        self.initial_base_noise = self.rng.normal(0.0, 1.0, self.signal_length)
        self.base_noise = self.initial_base_noise.copy()

        self.canvas = PlotCanvas()

        #Слайдери (sliders), які відповідають за амплітуду, частоту гармоніки, а також слайдери для параметрів шуму
        self.amplitude_slider = LabeledSlider(title="Amplitude", min_value=1, max_value=500, default_value=100, scale=100, decimals=2)

        self.frequency_slider = LabeledSlider(title="Frequency", min_value=1, max_value=2000, default_value=200, scale=100, decimals=2)

        self.phase_slider = LabeledSlider(title="Phase", min_value=-314, max_value=314, default_value=0, scale=100, decimals=2)

        self.noise_mean_slider = LabeledSlider(title="Noise Mean", min_value=-300, max_value=300, default_value=0, scale=100, decimals=2)

        self.noise_cov_slider = LabeledSlider(title="Noise Covariance", min_value=0, max_value=400, default_value=5, scale=100, decimals=2)

        self.cutoff_slider = LabeledSlider(title="Cutoff Frequency", min_value=1, max_value=10000, default_value=500, scale=100, decimals=2)

        self.order_slider = LabeledSlider(title="Filter Order", min_value=1, max_value=10, default_value=4, scale=1, decimals=0)

        #Чекбокс для перемикання відображення шуму на гармоніці
        self.show_noise_checkbox = QCheckBox("Показувати шум")
        self.show_noise_checkbox.setChecked(self.default_show_noise)

        #Додайте відповідні інтерактивні елементи (чекбокс показу, параметри фільтра тощо) та оновіть наявні: відфільтрована гармоніка має оновлюватись разом з початковою.
        self.enable_filter_checkbox = QCheckBox("Увімкнути фільтр")
        self.enable_filter_checkbox.setChecked(self.default_filter_enabled)

        #Кнопка «Reset», яка відновлює початкові параметри
        self.reset_button = QPushButton("Reset")
        self.new_noise_button = QPushButton("Згенерувати новий шум")

        self.build_ui()
        self.connect_signals()
        self.update_plot()

    def build_ui(self):
        central_widget = QWidget()
        self.setCentralWidget(central_widget)

        main_layout = QVBoxLayout(central_widget)
        main_layout.addWidget(self.canvas)

        harmonic_box = QGroupBox("Параметри гармоніки")
        harmonic_layout = QVBoxLayout(harmonic_box)
        harmonic_layout.addWidget(self.amplitude_slider)
        harmonic_layout.addWidget(self.frequency_slider)
        harmonic_layout.addWidget(self.phase_slider)

        noise_box = QGroupBox("Параметри шуму")
        noise_layout = QVBoxLayout(noise_box)
        noise_layout.addWidget(self.noise_mean_slider)
        noise_layout.addWidget(self.noise_cov_slider)
        noise_layout.addWidget(self.show_noise_checkbox)
        noise_layout.addWidget(self.new_noise_button)

        filter_box = QGroupBox("Параметри фільтра")
        filter_layout = QVBoxLayout(filter_box)
        filter_layout.addWidget(self.enable_filter_checkbox)
        filter_layout.addWidget(self.cutoff_slider)
        filter_layout.addWidget(self.order_slider)
        filter_layout.addWidget(self.reset_button)

        controls_grid = QGridLayout()
        controls_grid.addWidget(harmonic_box, 0, 0)
        controls_grid.addWidget(noise_box, 0, 1)
        controls_grid.addWidget(filter_box, 0, 2)

        main_layout.addLayout(controls_grid)

    def connect_signals(self):
        self.amplitude_slider.slider.valueChanged.connect(self.update_plot)
        self.frequency_slider.slider.valueChanged.connect(self.update_plot)
        self.phase_slider.slider.valueChanged.connect(self.update_plot)
        self.noise_mean_slider.slider.valueChanged.connect(self.update_plot)
        self.noise_cov_slider.slider.valueChanged.connect(self.update_plot)
        self.cutoff_slider.slider.valueChanged.connect(self.update_plot)
        self.order_slider.slider.valueChanged.connect(self.update_plot)

        self.show_noise_checkbox.stateChanged.connect(self.update_plot)
        self.enable_filter_checkbox.stateChanged.connect(self.update_plot)

        self.reset_button.clicked.connect(self.reset_values)
        self.new_noise_button.clicked.connect(self.generate_new_noise)

    def get_current_parameters(self):
        return {
            "amplitude": self.amplitude_slider.value(),
            "frequency": self.frequency_slider.value(),
            "phase": self.phase_slider.value(),
            "noise_mean": self.noise_mean_slider.value(),
            "noise_covariance": self.noise_cov_slider.value(),
            "show_noise": self.show_noise_checkbox.isChecked(),
            "filter_enabled": self.enable_filter_checkbox.isChecked(),
            "cutoff_frequency": self.cutoff_slider.value(),
            "filter_order": int(self.order_slider.value()),
        }

    def update_plot(self):
        params = self.get_current_parameters()

        t, y_clean, y_noisy, y_display = harmonic_with_noise(
            amplitude=params["amplitude"],
            frequency=params["frequency"],
            phase=params["phase"],
            noise_mean=params["noise_mean"],
            noise_covariance=params["noise_covariance"],
            show_noise=params["show_noise"],
            duration=self.duration,
            discretization_rate=self.discretization_rate,
            base_noise=self.base_noise,
        )

        if params["filter_enabled"]:
            y_filtered = apply_iir_lowpass(
                signal=y_noisy,
                cutoff_frequency=params["cutoff_frequency"],
                sample_rate=self.discretization_rate,
                filter_order=params["filter_order"],
            )
        else:
            y_filtered = y_noisy.copy()

        self.canvas.draw_plots(
            t=t,
            y_clean=y_clean,
            y_noisy=y_noisy,
            y_display=y_display,
            y_filtered=y_filtered,
            show_noise=params["show_noise"],
            filter_enabled=params["filter_enabled"],
        )

    def generate_new_noise(self):
        self.base_noise = self.rng.normal(0.0, 1.0, self.signal_length)
        self.update_plot()

    #Кнопка «Reset», яка відновлює початкові параметри
    def reset_values(self):
        widgets = [
            self.amplitude_slider.slider,
            self.frequency_slider.slider,
            self.phase_slider.slider,
            self.noise_mean_slider.slider,
            self.noise_cov_slider.slider,
            self.cutoff_slider.slider,
            self.order_slider.slider,
            self.show_noise_checkbox,
            self.enable_filter_checkbox,
        ]

        for widget in widgets:
            widget.blockSignals(True)

        self.amplitude_slider.set_value(self.default_amplitude)
        self.frequency_slider.set_value(self.default_frequency)
        self.phase_slider.set_value(self.default_phase)
        self.noise_mean_slider.set_value(self.default_noise_mean)
        self.noise_cov_slider.set_value(self.default_noise_covariance)
        self.cutoff_slider.set_value(self.default_cutoff_frequency)
        self.order_slider.set_value(self.default_filter_order)

        self.show_noise_checkbox.setChecked(self.default_show_noise)
        self.enable_filter_checkbox.setChecked(self.default_filter_enabled)

        for widget in widgets:
            widget.blockSignals(False)

        self.base_noise = self.initial_base_noise.copy()

        self.update_plot()

In [ ]:
app = QApplication(sys.argv)
window = MainWindow()
window.resize(1280, 720)
window.show()
sys.exit(app.exec())